# Census Data Engineering Pipeline

This notebook implements the Data Engineering layer from the provided problem statement for `census_data_raw.csv`.

Pipeline flow: ingestion contract, schema standardization, cleaning, timestamp parsing, label normalization, numeric casting, data quality metrics, Delta publish, and validation.

## 1. Configuration

Update `raw_path` if the CSV has been uploaded to a different Databricks location, for example `/FileStore/tables/census_data_raw.csv`.

In [ ]:
from pyspark.sql import functions as F, types as T
from functools import reduce
import re

raw_path = "../data/raw/census_data_raw.csv"
target_table = "workshop.default.donation_data_v1"
dq_table = "workshop.default.census_dq_metrics_v1"

expected_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income", "event_time", "random_flag",
    "source_system"
]


## 2. Ingestion Contract

Read the CSV with headers, keep all fields as strings initially, capture row count, and validate the column contract before applying transformations.

In [ ]:
raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(raw_path)
)

raw_row_count = raw_df.count()
raw_column_count = len(raw_df.columns)

metadata_df = spark.createDataFrame(
    [(raw_row_count, raw_column_count, raw_df.columns)],
    ["raw_row_count", "raw_column_count", "raw_columns"]
)

display(metadata_df)


## 3. Column Standardization

Convert names to lowercase, trim whitespace, replace punctuation with underscores, and enforce deterministic ordering.

In [ ]:
def standardize_column_name(name: str) -> str:
    cleaned = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned

standardized_columns = [standardize_column_name(c) for c in raw_df.columns]
df = raw_df.toDF(*standardized_columns)

missing_columns = sorted(set(expected_columns) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(expected_columns))

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df = df.select(*expected_columns)

schema_check_df = spark.createDataFrame(
    [(missing_columns, extra_columns, df.columns)],
    ["missing_columns", "extra_columns", "ordered_columns"]
)
display(schema_check_df)


## 4. Raw Data Quality Scan

Identify risky tokens before cleaning: empty strings, obvious error tokens, duplicate records, timestamp parse failures, and non-canonical income labels.

In [ ]:
def empty_to_null(col_name: str):
    value = F.trim(F.col(col_name).cast("string"))
    return F.when(value == "", None).otherwise(value)

trimmed_df = df.select([empty_to_null(c).alias(c) for c in df.columns])

duplicate_raw_count = raw_row_count - trimmed_df.dropDuplicates().count()
error_token_conditions = [F.lower(F.col(c).cast("string")).rlike("^error") for c in trimmed_df.columns]
error_token_count = trimmed_df.filter(reduce(lambda a, b: a | b, error_token_conditions)).count()

raw_scan_df = spark.createDataFrame(
    [(raw_row_count, duplicate_raw_count, error_token_count)],
    ["raw_row_count", "duplicate_raw_count", "error_token_count"]
)
display(raw_scan_df)


## 5. Cleaning Helpers

Centralize reusable parsing logic for numeric values, age, timestamps, and income labels.

In [ ]:
def clean_numeric(col_name: str):
    raw = F.lower(F.trim(F.col(col_name).cast("string")))
    no_commas = F.regexp_replace(raw, ",", "")
    multiplier = F.when(no_commas.rlike("k$"), F.lit(1000.0)).otherwise(F.lit(1.0))
    numeric_text = F.regexp_replace(no_commas, "[^0-9.-]", "")
    return F.when(raw.isNull() | (raw == "") | raw.rlike("^error"), None).otherwise(numeric_text.cast("double") * multiplier)

def clean_age(col_name: str):
    raw = F.lower(F.trim(F.col(col_name).cast("string")))
    digits = F.regexp_extract(raw, "([0-9]+)", 1)
    return F.when(raw.isNull() | (raw == "") | raw.rlike("^error") | (digits == ""), None).otherwise(digits.cast("int"))

def normalize_income(col_name: str):
    raw = F.lower(F.regexp_replace(F.trim(F.col(col_name).cast("string")), "\\s+", ""))
    return (
        F.when(raw.isNull() | (raw == ""), None)
        .when(raw.isin("<=80k", "=<80k", "le80k", "low", "0"), "<=80k")
        .when(raw.isin(">80k", "gt80k", "high", "1"), ">80k")
        .otherwise(None)
    )

def parse_event_time(col_name: str):
    raw = F.trim(F.col(col_name).cast("string"))
    epoch_seconds = F.when(raw.rlike("^[0-9]{10}$"), F.to_timestamp(F.from_unixtime(raw.cast("bigint"))))
    epoch_millis = F.when(raw.rlike("^[0-9]{13}$"), F.to_timestamp(F.from_unixtime((raw.cast("double") / F.lit(1000)).cast("bigint"))))
    return F.coalesce(
        F.to_timestamp(raw, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(raw, "yyyy-MM-dd'T'HH:mm:ss"),
        F.to_timestamp(raw, "yyyy-MM-dd"),
        F.to_timestamp(raw, "dd/MM/yy HH:mm"),
        F.to_timestamp(raw, "dd/MM/yy"),
        F.to_timestamp(raw, "MM/dd/yy HH:mm"),
        F.to_timestamp(raw, "MM/dd/yy"),
        epoch_seconds,
        epoch_millis
    )


## 6. Apply Cleaning Rules

Trim categorical fields, fill `workclass`, clean numeric fields, parse `event_time`, normalize labels, and retain raw values for auditability where useful.

In [ ]:
categorical_columns = [
    "workclass", "education_level", "marital_status", "occupation", "relationship",
    "race", "sex", "native_country", "random_flag", "source_system"
]

cleaned_df = trimmed_df

for c in categorical_columns:
    cleaned_df = cleaned_df.withColumn(c, empty_to_null(c))

cleaned_df = (
    cleaned_df
    .withColumn("age_raw", F.col("age"))
    .withColumn("education_num_raw", F.col("education_num"))
    .withColumn("capital_gain_raw", F.col("capital_gain"))
    .withColumn("capital_loss_raw", F.col("capital_loss"))
    .withColumn("hours_per_week_raw", F.col("hours_per_week"))
    .withColumn("income_raw", F.col("income"))
    .withColumn("event_time_raw", F.col("event_time"))
    .withColumn("age", clean_age("age"))
    .withColumn("education_num", clean_numeric("education_num"))
    .withColumn("capital_gain", clean_numeric("capital_gain"))
    .withColumn("capital_loss", clean_numeric("capital_loss"))
    .withColumn("hours_per_week", clean_numeric("hours_per_week"))
    .withColumn("workclass", F.coalesce(F.col("workclass"), F.lit("Unknown")))
    .withColumn("event_time_std", parse_event_time("event_time"))
    .withColumn("income_label", normalize_income("income"))
)

display(cleaned_df.limit(20))


## 7. Remove Duplicates

No explicit record id exists, so the business key uses stable citizen attributes plus event time, source, and normalized label.

In [ ]:
business_key_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "native_country", "event_time_std",
    "income_label", "source_system"
]

deduped_df = cleaned_df.dropDuplicates(business_key_columns)
deduped_row_count = deduped_df.count()
duplicate_business_key_count = cleaned_df.count() - deduped_row_count

display(spark.createDataFrame([(duplicate_business_key_count, deduped_row_count)], ["duplicate_business_key_count", "deduped_row_count"]))


## 8. Final Curated Dataset

Select the publish schema in deterministic order and add processing metadata.

In [ ]:
curated_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income_label", "event_time_std",
    "random_flag", "source_system", "age_raw", "income_raw", "event_time_raw"
]

curated_df = (
    deduped_df.select(*curated_columns)
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("pipeline_version", F.lit("v1"))
)

curated_df.printSchema()
display(curated_df.limit(20))


## 9. Data Quality Metrics

Capture row counts, invalid counts, cast failures, timestamp parse failures, label drift, and null counts.

In [ ]:
timestamp_parse_failures = cleaned_df.filter(F.col("event_time_raw").isNotNull() & F.col("event_time_std").isNull()).count()
label_failures = cleaned_df.filter(F.col("income_raw").isNotNull() & F.col("income_label").isNull()).count()
age_cast_failures = cleaned_df.filter(F.col("age_raw").isNotNull() & F.col("age").isNull()).count()

numeric_raw_column_map = {
    "education_num": "education_num_raw",
    "capital_gain": "capital_gain_raw",
    "capital_loss": "capital_loss_raw",
    "hours_per_week": "hours_per_week_raw",
}

numeric_cast_failure_count = 0
for clean_col, raw_col in numeric_raw_column_map.items():
    numeric_cast_failure_count += cleaned_df.filter(F.col(raw_col).isNotNull() & F.col(clean_col).isNull()).count()

metric_rows = [
    ("raw_row_count", raw_row_count),
    ("curated_row_count", curated_df.count()),
    ("raw_duplicate_count", duplicate_raw_count),
    ("business_key_duplicate_count", duplicate_business_key_count),
    ("error_token_count", error_token_count),
    ("timestamp_parse_failures", timestamp_parse_failures),
    ("label_failures", label_failures),
    ("age_cast_failures", age_cast_failures),
    ("numeric_cast_failure_count", numeric_cast_failure_count),
]

for c in curated_df.columns:
    metric_rows.append((f"null_count__{c}", curated_df.filter(F.col(c).isNull()).count()))

dq_metrics_df = (
    spark.createDataFrame(metric_rows, ["metric_name", "metric_value"])
    .withColumn("metric_value", F.col("metric_value").cast("long"))
    .withColumn("measured_at", F.current_timestamp())
    .withColumn("source_path", F.lit(raw_path))
)

display(dq_metrics_df.orderBy("metric_name"))


## 10. Validation Gates

Fail fast if the curated dataset has unexpected labels, no rows, or impossible numeric values.

In [ ]:
valid_labels = {"<=80k", ">80k"}
actual_labels = {row["income_label"] for row in curated_df.select("income_label").distinct().collect() if row["income_label"] is not None}

if not actual_labels.issubset(valid_labels):
    raise ValueError(f"Unexpected income labels found: {actual_labels - valid_labels}")

if curated_df.count() == 0:
    raise ValueError("Curated dataset is empty.")

invalid_age_count = curated_df.filter((F.col("age") < 0) | (F.col("age") > 120)).count()
invalid_hours_count = curated_df.filter((F.col("hours_per_week") < 0) | (F.col("hours_per_week") > 168)).count()

if invalid_age_count > 0:
    raise ValueError(f"Invalid age records found: {invalid_age_count}")

if invalid_hours_count > 0:
    raise ValueError(f"Invalid hours_per_week records found: {invalid_hours_count}")

print("Validation passed")


## 11. Publish to Delta

Write the curated data and data quality metrics using overwrite mode so the pipeline is idempotent.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workshop.default")

(
    curated_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

(
    dq_metrics_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_table)
)

print(f"Published curated table: {target_table}")
print(f"Published DQ metrics table: {dq_table}")


## 12. Publish Validation

Confirm the published Delta table row count and schema match the in-memory curated dataset.

In [ ]:
published_df = spark.table(target_table)
published_count = published_df.count()

if published_count != curated_df.count():
    raise ValueError(f"Published row count mismatch: {published_count} != {curated_df.count()}")

display(published_df.limit(20))
display(spark.table(dq_table).orderBy("metric_name"))
print("Publish validation passed")
